In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate

ROOT = Path("/Users/yufeizhou/Desktop/heat-exposure-compare")
OUT = ROOT / "outputs/pilot/alphaearth"

# =========================
# 1. paths
# =========================
city_cfg = {
    "Houston": {
        "embed_file": OUT / "houston_alphaearth_2024_tract_mean_mosaic_clean.csv",
        "heat_file": ROOT / "data_processed/houston/houston_master_with_lst_hi_fixed.gpkg",
        "pc_file": OUT / "houston_alphaearth_pc7_scores_corrected.csv",
    },
    "Phoenix": {
        "embed_file": OUT / "phoenix_alphaearth_2024_tract_mean_mosaic_clean.csv",
        "heat_file": ROOT / "data_processed/phoenix/phoenix_master_with_lst_hi_fixed.gpkg",
        "pc_file": OUT / "phoenix_alphaearth_pc7_scores_corrected.csv",
    },
}

selected_dims_file = OUT / "city_rawdim_selected_dims.csv"

# =========================
# 2. helpers
# =========================
def standardize_geoid(s):
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"\s+", "", regex=True)
    return s

def pick_first_existing(df, candidates, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise KeyError(f"Cannot find any of {candidates} in columns: {list(df.columns)}")
    return None

def read_heat_table(path):
    path = Path(path)
    if path.suffix.lower() == ".gpkg":
        df = gpd.read_file(path)
        if "geometry" in df.columns:
            df = df.drop(columns="geometry")
        return df.copy()
    elif path.suffix.lower() == ".csv":
        return pd.read_csv(path).copy()
    else:
        raise ValueError(f"Unsupported file type: {path}")

def coerce_numeric_inplace(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

def build_city_df(city_name, cfg, selected_dims_df):
    print(f"\n===== {city_name} =====")
    embed_file = Path(cfg["embed_file"])
    heat_file = Path(cfg["heat_file"])
    pc_file = Path(cfg["pc_file"])

    print("embed exists:", embed_file.exists(), embed_file)
    print("heat exists :", heat_file.exists(), heat_file)
    print("pc exists   :", pc_file.exists(), pc_file)

    if not embed_file.exists():
        raise FileNotFoundError(f"{city_name} embed file not found: {embed_file}")
    if not heat_file.exists():
        raise FileNotFoundError(f"{city_name} heat file not found: {heat_file}")
    if not pc_file.exists():
        raise FileNotFoundError(f"{city_name} pc file not found: {pc_file}")

    # read
    embed_df = pd.read_csv(embed_file, dtype={"GEOID": str}).copy()
    heat_df = read_heat_table(heat_file).copy()
    pc_df = pd.read_csv(pc_file, dtype={"GEOID": str}).copy()

    # GEOID
    geoid_embed = pick_first_existing(embed_df, ["GEOID", "geoid"])
    geoid_heat = pick_first_existing(heat_df, ["GEOID", "geoid"])
    geoid_pc = pick_first_existing(pc_df, ["GEOID", "geoid"])

    embed_df["GEOID"] = standardize_geoid(embed_df[geoid_embed])
    heat_df["GEOID"] = standardize_geoid(heat_df[geoid_heat])
    pc_df["GEOID"] = standardize_geoid(pc_df[geoid_pc])

    # pad to same width
    width = int(
        max(
            embed_df["GEOID"].str.len().max(),
            heat_df["GEOID"].str.len().max(),
            pc_df["GEOID"].str.len().max(),
        )
    )
    embed_df["GEOID"] = embed_df["GEOID"].str.zfill(width)
    heat_df["GEOID"] = heat_df["GEOID"].str.zfill(width)
    pc_df["GEOID"] = pc_df["GEOID"].str.zfill(width)

    # choose variables
    lst_col = pick_first_existing(heat_df, ["lst_c", "lst_summer_mean_c", "lst"])
    hi_col = pick_first_existing(heat_df, ["hi_c", "hi_summer_mean_c", "hi"], required=False)

    income_col = pick_first_existing(
        heat_df,
        ["median_household_income_clean", "median_household_income", "income"]
    )
    poverty_col = pick_first_existing(
        heat_df,
        ["poverty_rate", "poverty", "poverty_pct"],
        required=False
    )
    pop_col = pick_first_existing(
        heat_df,
        ["total_population_clean", "total_population", "population", "pop"]
    )
    aland_col = pick_first_existing(heat_df, ["ALAND", "aland"], required=False)
    awater_col = pick_first_existing(heat_df, ["AWATER", "awater"], required=False)

    # create hi_minus_lst if needed
    if "hi_minus_lst" not in heat_df.columns:
        if hi_col is not None and lst_col is not None:
            heat_df["hi_minus_lst"] = pd.to_numeric(heat_df[hi_col], errors="coerce") - pd.to_numeric(heat_df[lst_col], errors="coerce")

    # keep needed heat cols
    keep_heat = ["GEOID", lst_col, income_col, pop_col]
    if hi_col is not None:
        keep_heat.append(hi_col)
    if poverty_col is not None:
        keep_heat.append(poverty_col)
    if aland_col is not None:
        keep_heat.append(aland_col)
    if awater_col is not None:
        keep_heat.append(awater_col)
    if "hi_minus_lst" in heat_df.columns:
        keep_heat.append("hi_minus_lst")

    heat_df = heat_df[sorted(set(keep_heat))].copy()

    # numeric
    numeric_candidates = [c for c in heat_df.columns if c != "GEOID"]
    coerce_numeric_inplace(heat_df, numeric_candidates)

    # selected raw dims for this city
    city_dims = selected_dims_df.loc[selected_dims_df["city"] == city_name, "selected_raw_dim"].dropna().astype(str).tolist()
    city_dims = [d for d in city_dims if d in embed_df.columns]

    # merge
    df_city = embed_df[["GEOID"] + city_dims].merge(heat_df, on="GEOID", how="inner")
    df_city = df_city.merge(pc_df, on="GEOID", how="inner")

    print("embed rows:", len(embed_df))
    print("heat rows :", len(heat_df))
    print("pc rows   :", len(pc_df))
    print("merged rows:", len(df_city))

    # pcs
    pc_cols = [c for c in [f"PC{i}" for i in range(1, 8)] if c in df_city.columns]

    # baseline cols
    baseline_cols = [income_col, pop_col]
    if poverty_col is not None:
        baseline_cols.append(poverty_col)
    if aland_col is not None:
        baseline_cols.append(aland_col)
    if awater_col is not None:
        baseline_cols.append(awater_col)

    baseline_cols = [c for c in baseline_cols if c in df_city.columns]

    # target
    target_col = "hi_minus_lst"
    if target_col not in df_city.columns or df_city[target_col].isna().all():
        if hi_col is not None and lst_col is not None and hi_col in df_city.columns and lst_col in df_city.columns:
            df_city[target_col] = pd.to_numeric(df_city[hi_col], errors="coerce") - pd.to_numeric(df_city[lst_col], errors="coerce")

    print("target col:", target_col)
    print("target non-null:", df_city[target_col].notna().sum() if target_col in df_city.columns else 0)

    # force numeric for modeling columns
    model_numeric_cols = city_dims + baseline_cols + pc_cols + [target_col]
    model_numeric_cols = [c for c in model_numeric_cols if c in df_city.columns]
    coerce_numeric_inplace(df_city, model_numeric_cols)

    print("baseline cols:", baseline_cols)
    print("pc cols:", pc_cols)
    print("stable raw dims selected:", city_dims)

    # useful debug
    miss_df = pd.DataFrame({
        "column": model_numeric_cols,
        "missing_n": [df_city[c].isna().sum() for c in model_numeric_cols],
        "missing_pct": [df_city[c].isna().mean() for c in model_numeric_cols]
    }).sort_values("missing_pct", ascending=False)

    display(miss_df.head(15))

    return {
        "city": city_name,
        "df": df_city,
        "baseline_cols": baseline_cols,
        "pc_cols": pc_cols,
        "raw_dims": city_dims,
        "target_col": target_col,
    }

def eval_model(df, feature_cols, target_col, model_name, city_name):
    use_cols = [c for c in feature_cols if c in df.columns] + [target_col]
    sub = df[use_cols].dropna().copy()

    print(f"{city_name} / {model_name} usable rows after dropna:", len(sub))

    if len(sub) < 50:
        return {
            "city": city_name,
            "model": model_name,
            "n_rows": len(sub),
            "n_features": len(feature_cols),
            "cv_r2_mean": np.nan,
            "cv_r2_std": np.nan,
            "cv_rmse_mean": np.nan,
            "cv_rmse_std": np.nan,
            "cv_mae_mean": np.nan,
            "cv_mae_std": np.nan,
            "note": f"too_few_rows_after_dropna_{len(sub)}",
        }

    X = sub[[c for c in feature_cols if c in sub.columns]]
    y = sub[target_col]

    rf = RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scoring = {
        "r2": "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error"
    }

    res = cross_validate(
        rf, X, y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    return {
        "city": city_name,
        "model": model_name,
        "n_rows": len(sub),
        "n_features": X.shape[1],
        "cv_r2_mean": np.mean(res["test_r2"]),
        "cv_r2_std": np.std(res["test_r2"]),
        "cv_rmse_mean": -np.mean(res["test_rmse"]),
        "cv_rmse_std": np.std(-res["test_rmse"]),
        "cv_mae_mean": -np.mean(res["test_mae"]),
        "cv_mae_std": np.std(-res["test_mae"]),
        "note": "",
    }

# =========================
# 3. build city data
# =========================
selected_dims_df = pd.read_csv(selected_dims_file).copy()

city_data = {}
for city, cfg in city_cfg.items():
    city_data[city] = build_city_df(city, cfg, selected_dims_df)

# =========================
# 4. cross-city transfer
# =========================
rows = []

for target_city in ["Houston", "Phoenix"]:
    other_city = "Phoenix" if target_city == "Houston" else "Houston"

    df_city = city_data[target_city]["df"].copy()
    baseline_cols = city_data[target_city]["baseline_cols"]
    pc_cols = city_data[target_city]["pc_cols"]
    target_col = city_data[target_city]["target_col"]

    shared_dims = sorted(
        list(set(city_data["Houston"]["raw_dims"]).intersection(set(city_data["Phoenix"]["raw_dims"])))
    )
    other_city_only_dims = sorted(
        list(set(city_data[other_city]["raw_dims"]) - set(city_data[target_city]["raw_dims"]))
    )

    model_specs = [
        ("baseline", baseline_cols),
        ("baseline_plus_shared_dims_only", baseline_cols + shared_dims),
        ("baseline_plus_other_city_stable_dims", baseline_cols + other_city_only_dims),
    ]

    # 可选：如果你想加本城PC对照
    if len(pc_cols) > 0:
        model_specs.insert(1, ("baseline_plus_pc1_to_pc7", baseline_cols + pc_cols))

    print(f"\n===== evaluating {target_city} =====")
    print("target col:", target_col)

    for model_name, feats in model_specs:
        out = eval_model(df_city, feats, target_col, model_name, target_city)
        rows.append(out)
        print(model_name, "-> r2 =", out["cv_r2_mean"], "| n =", out["n_rows"], "| note =", out["note"])

summary_df = pd.DataFrame(rows)
summary_csv = OUT / "city_cross_city_transfer_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("\nSaved:")
print(summary_csv)
display(summary_df)


===== Houston =====
embed exists: True /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/houston_alphaearth_2024_tract_mean_mosaic_clean.csv
heat exists : True /Users/yufeizhou/Desktop/heat-exposure-compare/data_processed/houston/houston_master_with_lst_hi_fixed.gpkg
pc exists   : True /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/houston_alphaearth_pc7_scores_corrected.csv
embed rows: 654
heat rows : 654
pc rows   : 654
merged rows: 654
target col: hi_minus_lst
target non-null: 549
baseline cols: ['median_household_income_clean', 'total_population_clean', 'poverty_rate', 'ALAND', 'AWATER']
pc cols: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7']
stable raw dims selected: ['A02', 'A04', 'A07', 'A16', 'A17', 'A22', 'A25', 'A28', 'A31', 'A32', 'A34', 'A42', 'A51', 'A56', 'A59']


,column,missing_n,missing_pct
27,hi_minus_lst,105,0.160550
15,median_household_income_clean,9,0.013761
17,poverty_rate,3,0.004587
1,A04,0,0.000000
26,PC7,0,0.000000
25,PC6,0,0.000000
24,PC5,0,0.000000
23,PC4,0,0.000000
22,PC3,0,0.000000
21,PC2,0,0.000000



===== Phoenix =====
embed exists: True /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/phoenix_alphaearth_2024_tract_mean_mosaic_clean.csv
heat exists : True /Users/yufeizhou/Desktop/heat-exposure-compare/data_processed/phoenix/phoenix_master_with_lst_hi_fixed.gpkg
pc exists   : True /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/phoenix_alphaearth_pc7_scores_corrected.csv
embed rows: 373
heat rows : 373
pc rows   : 373
merged rows: 373
target col: hi_minus_lst
target non-null: 332
baseline cols: ['median_household_income_clean', 'total_population_clean', 'poverty_rate', 'ALAND', 'AWATER']
pc cols: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7']
stable raw dims selected: ['A01', 'A03', 'A04', 'A09', 'A19', 'A22', 'A26', 'A40', 'A44', 'A47', 'A52', 'A58', 'A59', 'A61', 'A62', 'A63']


,column,missing_n,missing_pct
28,hi_minus_lst,41,0.109920
16,median_household_income_clean,5,0.013405
18,poverty_rate,3,0.008043
15,A63,0,0.000000
27,PC7,0,0.000000
26,PC6,0,0.000000
25,PC5,0,0.000000
24,PC4,0,0.000000
23,PC3,0,0.000000
22,PC2,0,0.000000



===== evaluating Houston =====
target col: hi_minus_lst
Houston / baseline usable rows after dropna: 541
baseline -> r2 = 0.39804537805303825 | n = 541 | note = 
Houston / baseline_plus_pc1_to_pc7 usable rows after dropna: 541
baseline_plus_pc1_to_pc7 -> r2 = 0.7848992912733052 | n = 541 | note = 
Houston / baseline_plus_shared_dims_only usable rows after dropna: 541
baseline_plus_shared_dims_only -> r2 = 0.6163564098547993 | n = 541 | note = 
Houston / baseline_plus_other_city_stable_dims usable rows after dropna: 541
baseline_plus_other_city_stable_dims -> r2 = 0.39804537805303813 | n = 541 | note = 

===== evaluating Phoenix =====
target col: hi_minus_lst
Phoenix / baseline usable rows after dropna: 329
baseline -> r2 = 0.1795343402585407 | n = 329 | note = 
Phoenix / baseline_plus_pc1_to_pc7 usable rows after dropna: 329
baseline_plus_pc1_to_pc7 -> r2 = 0.6676631925706256 | n = 329 | note = 
Phoenix / baseline_plus_shared_dims_only usable rows after dropna: 329
baseline_plus_share

,city,model,n_rows,n_features,cv_r2_mean,cv_r2_std,cv_rmse_mean,cv_rmse_std,cv_mae_mean,cv_mae_std,note
0,Houston,baseline,541,5,0.398045,0.139796,2.140495,0.184304,1.566697,0.089742,
1,Houston,baseline_plus_pc1_to_pc7,541,12,0.784899,0.065437,1.283161,0.197907,0.921733,0.079648,
2,Houston,baseline_plus_shared_dims_only,541,8,0.616356,0.074460,1.716711,0.142039,1.269982,0.047018,
3,Houston,baseline_plus_other_city_stable_dims,541,5,0.398045,0.139796,2.140495,0.184304,1.566697,0.089742,
4,Phoenix,baseline,329,5,0.179534,0.110629,1.503657,0.081877,1.205079,0.069432,
5,Phoenix,baseline_plus_pc1_to_pc7,329,12,0.667663,0.057883,0.954644,0.060523,0.747565,0.058081,
6,Phoenix,baseline_plus_shared_dims_only,329,8,0.608283,0.073659,1.036095,0.080542,0.826975,0.085787,
7,Phoenix,baseline_plus_other_city_stable_dims,329,5,0.179534,0.110629,1.503657,0.081877,1.205079,0.069432,
